# 한국 우량 우선주 vs 보통주 10년 비교 · 괴리율 백분위 판독기 (Google Colab)

**목적:** 국내 대표 배당 우선주와 본주(보통주)를 **최근 10년 실제 시세·배당 기준**으로 비교하여,
1. **10년 장기 보유 시 보통주 vs 우선주 총수익률(배당 포함)** 대결 결과
2. **현재 가격비율(`우선주 ÷ 보통주`)과 괴리율(`1 − 가격비율`)이 지난 10년 중 상위 몇 %로 싼지/비싼지**
3. **신규 적립 시 지금 가격에서 우선주가 유리한지, 본주가 유리한지**
를 **API 키 없이** 원클릭으로 진단하고 A4 PDF/HTML 리포트를 Google Drive에 저장합니다.

### 자동 개인화 (비공개 Drive 설정 연동)
- Google Drive에 기존 비공개 설정 파일(`MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml`)이 있으면 **보유 중인 우선주에 자동으로 「보유중」 배지**를 붙여 최상단에 우선 표시합니다. *(공개 저장소에는 개인 보유 정보가 전혀 포함되지 않습니다.)*
- `PREF_CONFIG["CUSTOM_THRESHOLDS"]`에 본인만의 재검토 기준선(예: 현대차3우B 10년 중앙값 `0.559`)을 넣어두면 차트와 판정표에 기준선 돌파 여부가 자동 표시됩니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트(fonts-nanum) 및 PDF 도구 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, re, io, math, time, base64, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint", "pyyaml"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    import yaml
except ImportError:
    yaml = None

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")

In [ ]:
# ==============================================================================
# 2. 추적 대상 우선주 페어 설정 · Google Drive 연결 (비공개 보유종목 자동 태그)
# ==============================================================================
PREF_CONFIG = {
    # 추적할 (우선주 티커, 보통주 티커, 우선주명, 보통주명, 정관 추가배당 원)
    "PAIRS": [
        ("005935", "005930", "삼성전자우",   "삼성전자",     1),
        ("005389", "005380", "현대차3우B",   "현대차",      50),
        ("005387", "005380", "현대차2우B",   "현대차",     100),
        ("005385", "005380", "현대차우",     "현대차",      50),
        ("000815", "000810", "삼성화재우",   "삼성화재",     5),
        ("003555", "003550", "LG우",         "LG",          50),
        ("051915", "051910", "LG화학우",     "LG화학",      50),
        ("051905", "051900", "LG생활건강우", "LG생활건강",  50),
        ("071055", "071050", "한국금융지주우","한국금융지주", 50),
        ("005945", "005940", "NH투자증권우", "NH투자증권",  50),
    ],

    # 선택: 특정 우선주의 개인 재검토 가격비율 기준선 (우선주 ÷ 보통주)
    # 예: 현대차3우B 가격비율이 0.559 위로 올라가면 '할인 축소(재검토선 돌파)' 알림 표시
    "CUSTOM_THRESHOLDS": {
        "005389": {"ratio_high": 0.559, "note": "10년 중앙값 0.559 초과 시 교체 재검토"},
    },

    "LOOKBACK_YEARS": 10,          # 괴리율 분포 및 총수익률 비교 기간 (년)
    "MIN_LIQ_EOK": 5.0,            # 우선주 60일 일평균 거래대금 최소 기준 (억 원)

    # Google Drive 저장 및 비공개 포트폴리오 연동
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_preferred",
    "PORTFOLIO_YAML": "/content/drive/MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml",
}

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
HELD_PREF_SET = set()

if IN_COLAB and PREF_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(PREF_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] 리포트 저장 경로: {DRIVE_REPORT_DIR}")

        pf_p = Path(PREF_CONFIG["PORTFOLIO_YAML"])
        if yaml is not None and pf_p.exists():
            pf_obj = yaml.safe_load(pf_p.read_text(encoding="utf-8")) or {}
            for tk, info in (pf_obj.get("overrides") or {}).items():
                if isinstance(info, dict) and info.get("common_ticker"):
                    HELD_PREF_SET.add(str(tk).zfill(6))
            if HELD_PREF_SET:
                print(f"[보유 우선주 자동 감지] {sorted(HELD_PREF_SET)} -> 리포트 최상단 강조 표시")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 저장으로 진행합니다 ({type(e).__name__}: {e})")

print(f"[설정 완료] 총 {len(PREF_CONFIG['PAIRS'])}개 우선주-보통주 페어 | 기준 기간 {PREF_CONFIG['LOOKBACK_YEARS']}년")

In [ ]:
# ==============================================================================
# 3. 10년 일별 수정주가(네이버 fchart) + 10년 배당 포함 총수익률(Yahoo Finance) 수집
# ==============================================================================
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-preferred-tracker)"}

def fetch_naver_daily(ticker: str, count: int = 3000) -> pd.DataFrame:
    for attempt in range(3):
        try:
            r = requests.get(
                _FCHART_URL,
                params={"symbol": ticker, "timeframe": "day", "count": count, "requestType": 0},
                headers=_HEADERS,
                timeout=20,
            )
            r.raise_for_status()
            rows = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                parts = m.group(1).split("|")
                c, v = float(parts[4]), float(parts[5])
                if c > 0:
                    rows.append((pd.Timestamp(parts[0]), c, v))
            if rows:
                df = pd.DataFrame(rows, columns=["date", "close", "volume"]).set_index("date").sort_index()
                return df[~df.index.duplicated(keep="last")]
        except Exception:
            time.sleep(0.35 * (attempt + 1))
    return pd.DataFrame()

def fetch_yf_tr_and_div(ticker: str) -> tuple[pd.Series, float | None]:
    """Yahoo Finance .KS 10년 수정종가(배당재투자 TR) & 최근 1년 현금배당 합계 조회."""
    try:
        r = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}.KS",
            params={"range": "10y", "interval": "1d", "events": "div"},
            headers=_HEADERS,
            timeout=20,
        )
        r.raise_for_status()
        res = r.json()["chart"]["result"][0]
        ts = res["timestamp"]
        adj = res["indicators"].get("adjclose", [{}])[0].get("adjclose")
        vals = adj if adj is not None else res["indicators"]["quote"][0]["close"]
        idx = [pd.Timestamp(datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d")) for t in ts]
        s = pd.Series(vals, index=idx, dtype=float).dropna()
        s = s[~s.index.duplicated(keep="last")].sort_index()

        # 최근 12개월 현금배당 합계 (Yahoo div events 기준)
        divs = res.get("events", {}).get("dividends", {})
        if divs:
            latest_ts = max(ts)
            cutoff = latest_ts - 375 * 86400
            ttm_div = sum(float(v.get("amount", 0.0)) for k, v in divs.items() if int(k) >= cutoff)
        else:
            ttm_div = None
        return s, (ttm_div if (ttm_div and ttm_div > 0) else None)
    except Exception:
        return pd.Series(dtype=float), None

all_tks = sorted({tk for p in PREF_CONFIG["PAIRS"] for tk in (p[0], p[1])})
print(f"[시세 수집] {len(all_tks)}개 종목 네이버 3,000일 일봉 + Yahoo 10년 TR/배당 수집 중...")
NAVER_MAP = {}
YF_TR_MAP = {}
YF_DIV_MAP = {}

with ThreadPoolExecutor(max_workers=8) as ex:
    f_nav = {ex.submit(fetch_naver_daily, tk): tk for tk in all_tks}
    f_yf = {ex.submit(fetch_yf_tr_and_div, tk): tk for tk in all_tks}
    for fut in as_completed(f_nav):
        NAVER_MAP[f_nav[fut]] = fut.result()
    for fut in as_completed(f_yf):
        s_tr, ttm_d = fut.result()
        YF_TR_MAP[f_yf[fut]] = s_tr
        YF_DIV_MAP[f_yf[fut]] = ttm_d

print(f"[완료] 네이버 일봉 {sum(not d.empty for d in NAVER_MAP.values())}/{len(all_tks)} | Yahoo 10Y TR {sum(len(s)>100 for s in YF_TR_MAP.values())}/{len(all_tks)}")

In [ ]:
# ==============================================================================
# 4. 괴리율 10년 백분위 · 배당수익률 스프레드 · 10년 TR 대결 · 매수 판독
# ==============================================================================
KST = timezone(timedelta(hours=9))
as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

PAIR_RESULTS = []
RATIO_SERIES = {}

for p_tk, c_tk, p_nm, c_nm, extra_dps in PREF_CONFIG["PAIRS"]:
    df_p, df_c = NAVER_MAP.get(p_tk), NAVER_MAP.get(c_tk)
    if df_p is None or df_c is None or df_p.empty or df_c.empty:
        continue

    common_idx = df_p.index.intersection(df_c.index)
    end_dt = common_idx[-1]
    start_10y = end_dt - pd.Timedelta(days=int(365.25 * PREF_CONFIG["LOOKBACK_YEARS"]))
    sub_idx = common_idx[common_idx >= start_10y]
    if len(sub_idx) < 200:
        continue

    p_cl = df_p.loc[sub_idx, "close"]
    c_cl = df_c.loc[sub_idx, "close"]
    ratio = (p_cl / c_cl).dropna()
    RATIO_SERIES[p_tk] = ratio

    cur_p = float(p_cl.iloc[-1])
    cur_c = float(c_cl.iloc[-1])
    cur_ratio = float(ratio.iloc[-1])
    cur_disc = 1.0 - cur_ratio
    med_ratio = float(ratio.median())
    q25_ratio = float(ratio.quantile(0.25))
    q75_ratio = float(ratio.quantile(0.75))
    # 현재 가격비율의 10년 백분위 (0% = 10년 최저/가장 쌈, 100% = 10년 최고/가장 비쌈)
    pct_pos = float((ratio <= cur_ratio).mean() * 100.0)

    # 최근 60일 일평균 거래대금 (억 원)
    liq_p_eok = float((df_p["close"] * df_p["volume"]).tail(60).mean()) / 1e8
    liq_c_eok = float((df_c["close"] * df_c["volume"]).tail(60).mean()) / 1e8

    # 최근 1년 배당수익률 추정 (보통주 TTM 배당 기준 + 정관 추가배당 반영)
    div_c = YF_DIV_MAP.get(c_tk)
    div_p = YF_DIV_MAP.get(p_tk)
    if div_p is None and div_c is not None:
        div_p = div_c + extra_dps
    yld_c = (div_c / cur_c) if (div_c and cur_c > 0) else None
    yld_p = (div_p / cur_p) if (div_p and cur_p > 0) else None
    yld_spread = (yld_p - yld_c) if (yld_p is not None and yld_c is not None) else None

    # 10년 총수익률(TR, 배당 재투자 포함) 비교 — Yahoo 10Y adjclose 기준
    tr_p_s, tr_c_s = YF_TR_MAP.get(p_tk), YF_TR_MAP.get(c_tk)
    tr_p_cagr = tr_c_cagr = tr_diff = None
    if tr_p_s is not None and tr_c_s is not None and len(tr_p_s) > 250 and len(tr_c_s) > 250:
        t_idx = tr_p_s.index.intersection(tr_c_s.index)
        t_idx = t_idx[t_idx >= start_10y]
        if len(t_idx) > 250:
            yrs = (t_idx[-1] - t_idx[0]).days / 365.25
            tr_p_cagr = float((tr_p_s.loc[t_idx[-1]] / tr_p_s.loc[t_idx[0]]) ** (1.0 / yrs) - 1.0)
            tr_c_cagr = float((tr_c_s.loc[t_idx[-1]] / tr_c_s.loc[t_idx[0]]) ** (1.0 / yrs) - 1.0)
            tr_diff = tr_p_cagr - tr_c_cagr

    # 커스텀 재검토선 체크
    cust = PREF_CONFIG["CUSTOM_THRESHOLDS"].get(p_tk)
    cust_alert = ""
    if cust and cur_ratio > cust["ratio_high"]:
        cust_alert = f"기준선({cust['ratio_high']:.3f}) 돌파!"
    elif cust:
        cust_alert = f"기준선({cust['ratio_high']:.3f}) 하회(유지)"

    # 매수 판독
    if liq_p_eok < PREF_CONFIG["MIN_LIQ_EOK"]:
        verdict = "유동성 주의"
    elif pct_pos <= 25.0:
        verdict = "우선주 매우 유리"
    elif pct_pos <= 50.0:
        verdict = "우선주 유리"
    elif pct_pos <= 70.0:
        verdict = "중립(중앙값 부근)"
    else:
        verdict = "보통주 검토(할인축소)"

    is_held = p_tk in HELD_PREF_SET
    PAIR_RESULTS.append({
        "is_held": is_held,
        "구분": "보유중" if is_held else "관찰",
        "우선주": f"{p_nm}({p_tk})",
        "보통주": c_nm,
        "p_tk": p_tk, "p_nm": p_nm, "c_nm": c_nm,
        "cur_ratio": cur_ratio, "med_ratio": med_ratio, "q25_ratio": q25_ratio, "q75_ratio": q75_ratio,
        "min_ratio": float(ratio.min()), "max_ratio": float(ratio.max()),
        "pct_pos": pct_pos, "cur_disc": cur_disc,
        "liq_p_eok": liq_p_eok,
        "yld_p": yld_p, "yld_c": yld_c, "yld_spread": yld_spread,
        "tr_p_cagr": tr_p_cagr, "tr_c_cagr": tr_c_cagr, "tr_diff": tr_diff,
        "cust_alert": cust_alert,
        "판정": verdict,
    })

# 보유중 최상단 -> 10년 백분위 낮은 순(우선주가 역사적으로 싼 순) 정렬
PAIR_RESULTS.sort(key=lambda x: (not x["is_held"], x["pct_pos"]))

SUMMARY_DF = pd.DataFrame([
    {
        "구분": r["구분"],
        "우선주": r["우선주"],
        "현재 비율(우/보)": f"{r['cur_ratio']:.3f}",
        "10년 중앙값": f"{r['med_ratio']:.3f}",
        "현재 할인율": f"{r['cur_disc']:.1%}",
        "10년 백분위": f"{r['pct_pos']:.0f}%",
        "우선주 배당률": "-" if r["yld_p"] is None else f"{r['yld_p']:.2%}",
        "배당격차(우-보)": "-" if r["yld_spread"] is None else f"{r['yld_spread']*100:+.2f}%p",
        "10Y TR(우 vs 보)": "-" if r["tr_p_cagr"] is None else f"{r['tr_p_cagr']:+.1%} vs {r['tr_c_cagr']:+.1%}",
        "60일 거래대금": f"{r['liq_p_eok']:.1f}억",
        "판정": r["판정"],
        "비고": r["cust_alert"],
    }
    for r in PAIR_RESULTS
])

print(f"■ [우선주 vs 보통주 10년 괴리율 · 총수익률 종합 판독 ({as_of_str})]")
display(SUMMARY_DF)

In [ ]:
# ==============================================================================
# 5. 3패널 시각화 대시보드 + A4 PDF/HTML 리포트 Drive 자동 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64: str) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception:
        pass
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            img = mpimg.imread(io.BytesIO(base64.b64decode(chart_b64)), format="png")
            fig_p = plt.figure(figsize=(11.69, 8.27))
            plt.imshow(img); plt.axis("off"); plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception:
        return False

# 대시보드 1: 전체 우선주 10년 가격비율 범위 vs 현재 위치 + 10년 TR 격차
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.4, 5.8), facecolor="white")
fig.suptitle(f"한국 우량 우선주 10년 괴리율 위치 & 보통주 대비 총수익률 ({as_of_str})", fontsize=14.5, fontweight="bold", color="#0F2922", y=0.99)

ys = np.arange(len(PAIR_RESULTS))
labels = [f"{'★ ' if r['is_held'] else ''}{r['p_nm']}" for r in PAIR_RESULTS]
for i, r in enumerate(PAIR_RESULTS):
    ax1.plot([r["min_ratio"], r["max_ratio"]], [i, i], color="#CBD5E1", lw=2.5, zorder=1)
    ax1.plot([r["q25_ratio"], r["q75_ratio"]], [i, i], color="#64748B", lw=6.5, alpha=0.65, zorder=2, label="10년 25~75% 구간" if i == 0 else "")
    ax1.scatter(r["med_ratio"], i, color="#0F2922", marker="|", s=140, lw=2.2, zorder=3, label="10년 중앙값" if i == 0 else "")
    pt_col = "#059669" if r["pct_pos"] <= 35 else ("#DC2626" if r["pct_pos"] >= 70 else "#D97706")
    ax1.scatter(r["cur_ratio"], i, color=pt_col, s=95, edgecolors="#0F172A", lw=0.8, zorder=4, label="현재 가격비율" if i == 0 else "")
    ax1.text(r["max_ratio"] + 0.015, i, f"{r['cur_ratio']:.2f} ({r['pct_pos']:.0f}%)", va="center", fontsize=8.3, fontweight="bold", color=pt_col)
ax1.set_yticks(ys); ax1.set_yticklabels(labels, fontsize=9.0)
ax1.invert_yaxis()
ax1.set_xlabel("가격비율 (우선주 ÷ 보통주) — 왼쪽일수록 우선주 저렴")
ax1.set_title("① 10년 가격비율 범위 대비 현재 위치 (녹색=저평가 · 적색=할인축소)", fontsize=10.8, fontweight="bold", color="#0F2922", loc="left")
ax1.set_xlim(0.25, 1.08)
ax1.legend(loc="lower right", fontsize=8.0, frameon=True, facecolor="white")
ax1.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.7); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# (2) 10년 연평균 총수익률 격차 (우선주 TR - 보통주 TR)
tr_diffs = [(r["tr_diff"] * 100 if r["tr_diff"] is not None else 0.0) for r in PAIR_RESULTS]
bar_cols = ["#059669" if v >= 0 else "#EF4444" for v in tr_diffs]
bars = ax2.barh(ys, tr_diffs, color=bar_cols, height=0.58)
ax2.axvline(0, color="#334155", lw=1.0)
for b, v, r in zip(bars, tr_diffs, PAIR_RESULTS):
    txt = "자료부족" if r["tr_diff"] is None else f"{v:+.2f}%p/년"
    ax2.text(v + (0.08 if v >= 0 else -0.08), b.get_y() + b.get_height() / 2, txt, va="center", ha="left" if v >= 0 else "right", fontsize=8.3, fontweight="bold")
ax2.set_yticks(ys); ax2.set_yticklabels(labels, fontsize=9.0)
ax2.invert_yaxis()
ax2.set_xlabel("연평균 총수익률 격차 (우선주 TR − 보통주 TR, %p/년)")
ax2.set_title("② 최근 10년 총수익률(배당재투자) 대결: 우선주 vs 보통주", fontsize=10.8, fontweight="bold", color="#0F2922", loc="left")
ax2.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.7); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.94])
buf1 = io.BytesIO(); fig.savefig(buf1, format="png", dpi=150, bbox_inches="tight", facecolor="white")
CHART1_B64 = base64.b64encode(buf1.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig)

# 대시보드 2: 핵심 우선주 4종(보유주 최우선) 10년 가격비율 시계열 추이
focus_list = PAIR_RESULTS[:4]
fig2, axes = plt.subplots(2, 2, figsize=(15.4, 7.6), facecolor="white")
fig2.suptitle("핵심 우선주 4종 최근 10년 가격비율(우선주 ÷ 보통주) 추이 & 중앙값 밴드", fontsize=14.0, fontweight="bold", color="#0F2922", y=0.985)
for ax, r in zip(axes.flat, focus_list):
    s = RATIO_SERIES[r["p_tk"]]
    ax.plot(s.index, s, color="#0F766E", lw=1.4, label=f"가격비율 (현재 {r['cur_ratio']:.3f})")
    ax.axhline(r["med_ratio"], color="#0F2922", ls="--", lw=1.1, label=f"10년 중앙값 {r['med_ratio']:.3f}")
    ax.axhspan(r["q25_ratio"], r["q75_ratio"], color="#CBD5E1", alpha=0.35, label="25~75% 구간")
    cust = PREF_CONFIG["CUSTOM_THRESHOLDS"].get(r["p_tk"])
    if cust:
        ax.axhline(cust["ratio_high"], color="#DC2626", ls=":", lw=1.5, label=f"재검토선 {cust['ratio_high']:.3f}")
    ax.set_title(f"{'★ ' if r['is_held'] else ''}{r['p_nm']} / {r['c_nm']} — 백분위 {r['pct_pos']:.0f}% ({r['판정']})", fontsize=10.5, fontweight="bold", color="#0F2922", loc="left")
    ax.legend(loc="upper left", fontsize=7.8, frameon=True, facecolor="white", ncol=2)
    ax.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig2.tight_layout(rect=[0, 0, 1, 0.94])
buf2 = io.BytesIO(); fig2.savefig(buf2, format="png", dpi=150, bbox_inches="tight", facecolor="white")
CHART2_B64 = base64.b64encode(buf2.getvalue()).decode()
if IN_COLAB: plt.show()
plt.close(fig2)

def render_tbl(df: pd.DataFrame) -> str:
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    trs = ["<tr>" + "".join(f'<td class="{"col-left" if i<=1 else ""}">{escape(str(r[c]))}</td>' for i, c in enumerate(df.columns)) + "</tr>" for _, r in df.iterrows()]
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(trs)}</tbody></table>'

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>우선주 괴리율·총수익 판독 리포트 ({as_of_str})</title>
<style>
  @page {{ size: A4 landscape; margin: 10mm; }}
  body {{ margin:0; padding:12px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.5px; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:16px 20px; border-radius:10px; margin-bottom:12px; border-bottom:4px solid #D4A34A; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:12px 14px; margin-bottom:10px; break-inside:avoid; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.1px; }}
  table.tbl th {{ background:#F1F5F9; padding:5px; border-top:1.5px solid #0F2922; border-bottom:1px solid #CBD5E1; text-align:right; }}
  table.tbl td {{ padding:5px; border-bottom:1px solid #EEF2F6; text-align:right; }}
  table.tbl td.col-left, table.tbl th:first-child, table.tbl th:nth-child(2) {{ text-align:left; font-weight:700; }}
  img.chart {{ width:100%; height:auto; display:block; }}
  .page-break {{ page-break-before:always; break-before:page; }}
</style></head><body><div class="wrap">
  <div class="head">
    <h1 style="margin:0 0 4px;font-size:18px;">한국 우량 우선주 vs 보통주 10년 괴리율 · 총수익률 리포트 ({as_of_str})</h1>
    <div>네이버 3,000일 수정주가 + Yahoo Finance 10년 배당재투자 TR 기준 · ★ 표시 = 보유 우선주</div>
  </div>
  <div class="card"><h2 style="margin:0 0 8px;">01. 우선주 괴리율 · 배당격차 · 10년 총수익 종합표</h2>{render_tbl(SUMMARY_DF)}</div>
  <div class="card page-break"><h2 style="margin:0 0 8px;">02. 10년 괴리율 위치 &amp; 우선주 vs 보통주 총수익 대결</h2><img class="chart" src="data:image/png;base64,{CHART1_B64}"></div>
  <div class="card page-break"><h2 style="margin:0 0 8px;">03. 핵심 우선주 4종 10년 가격비율 시계열</h2><img class="chart" src="data:image/png;base64,{CHART2_B64}"></div>
</div></body></html>"""

r_html = f"preferred_tracker_report_{run_stamp}.html"
r_pdf = f"preferred_tracker_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, r_pdf, CHART1_B64)
print(f"[리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else '건너뜀'}")

# Google Drive 자동 연결 재시도 (Cell 2에서 미연결/취소된 경우에도 Cell 5에서 다시 시도)
if IN_COLAB and PREF_CONFIG.get("USE_GOOGLE_DRIVE", True) and DRIVE_REPORT_DIR is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(PREF_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 재연결 완료] {DRIVE_REPORT_DIR}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬(/content/{r_pdf if pdf_ok else r_html})에만 저장됨 ({type(e).__name__}: {e})")

if DRIVE_REPORT_DIR is not None:
    try:
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        (DRIVE_REPORT_DIR / "preferred_tracker_report_latest.html").write_text(html_report, encoding="utf-8")
        saved_targets = [DRIVE_REPORT_DIR / r_html]
        if pdf_ok and Path(r_pdf).exists():
            pdf_bytes = Path(r_pdf).read_bytes()
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(pdf_bytes)
            (DRIVE_REPORT_DIR / "preferred_tracker_report_latest.pdf").write_bytes(pdf_bytes)
            saved_targets.insert(0, DRIVE_REPORT_DIR / r_pdf)
        csv_name = f"preferred_summary_{run_stamp}.csv"
        SUMMARY_DF.to_csv(DRIVE_REPORT_DIR / csv_name, index=False, encoding="utf-8-sig")
        SUMMARY_DF.to_csv(DRIVE_REPORT_DIR / "preferred_summary_latest.csv", index=False, encoding="utf-8-sig")
        saved_targets.append(DRIVE_REPORT_DIR / csv_name)
        print(f"[Drive 저장 검증 완료] 폴더: {DRIVE_REPORT_DIR}")
        for fp in saved_targets:
            print(f"  - {fp.name} ({fp.stat().st_size/1024:.1f} KB)")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")